In [39]:
import polars as pl
from pathlib import Path

GTFS_DIRS = [
    # "raw/gtfs/brooklyn",
    "raw/gtfs/manhattan",
    # "raw/gtfs/queens"
]

def load_gtfs(filename):
    dfs = []

    for folder in GTFS_DIRS:
        path = Path(folder) / filename
        dfs.append(pl.read_csv(path))

    return pl.concat(dfs, how="vertical_relaxed")

In [40]:
routes = load_gtfs("routes.txt")
trips = load_gtfs("trips.txt")
stops = load_gtfs("stops.txt")
stop_times = load_gtfs("stop_times.txt")
shapes = load_gtfs("shapes.txt")
calendar = load_gtfs("calendar.txt")
calendar_dates=load_gtfs("calendar_dates.txt")

In [41]:
routes = routes.unique(
    subset=["route_id"]
)

trips = trips.unique(
    subset=["trip_id"]
)

stop_times = stop_times.unique(
    subset=[
        "trip_id",
        "stop_sequence",
        "stop_id"
    ]
)

stops = stops.unique(
    subset=["stop_id"]
)

shapes = shapes.unique(
    subset=[
        "shape_id",
        "shape_pt_sequence"
    ]
)

calendar = calendar.unique(
    subset=["service_id"]
)
calendar_dates = calendar_dates.unique(
    subset=["service_id", "date", "exception_type"]
)

In [42]:
TARGET_ROUTES = [
    # "Q17",
    # "Q24",
    # "Q27",
    # "Q30",
    # "Q31",
    "M1",
    "M2",
    "M4",
    "M15",
    "M101"
]

In [43]:
routes = routes.filter(
    pl.col("route_short_name").is_in(TARGET_ROUTES)
)

In [44]:
routes

route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_color,route_text_color
str,str,str,str,str,i64,str,str
"""M4""","""MTA NYCT""","""M4""","""The Cloisters - 32 St""","""via 5th Av / Madison Av / Broa…",3,"""00AEEF""","""FFFFFF"""
"""M2""","""MTA NYCT""","""M2""","""Washington Heights - East Vill…","""via 5th Av / Madison Av / AC P…",3,"""B933AD""","""FFFFFF"""
"""M101""","""MTA NYCT""","""M101""","""East Village - Fort George""","""via Third Av / Lexington Av / …",3,"""FAA61A""","""FFFFFF"""
"""M15""","""MTA NYCT""","""M15""","""East Harlem - South Ferry""","""via 1st Av / 2nd Av""",3,"""006CB7""","""FFFFFF"""
"""M1""","""MTA NYCT""","""M1""","""Harlem - East Village""","""via 5th Av / Madison Av""",3,"""EE352E""","""FFFFFF"""


In [45]:
trips = trips.join(
    routes.select("route_id"),
    on="route_id",
    how="inner"
)

In [46]:
stop_times = stop_times.join(
    trips.select("trip_id"),
    on="trip_id",
    how="inner"
)

In [47]:
stops = stops.join(
    stop_times.select("stop_id").unique(),
    on="stop_id",
    how="inner"
)

In [48]:
shapes = shapes.join(
    trips.select("shape_id").unique(),
    on="shape_id",
    how="inner"
)

In [50]:
stops = stops.with_columns([
    pl.col("stop_lat")
      .str.strip_chars()
      .cast(pl.Float64),

    pl.col("stop_lon")
      .str.strip_chars()
      .cast(pl.Float64),
])

In [51]:
print(routes.schema)
print(trips.schema)
print(stop_times.schema)
print(stops.schema)
print(shapes.schema)

Schema([('route_id', String), ('agency_id', String), ('route_short_name', String), ('route_long_name', String), ('route_desc', String), ('route_type', Int64), ('route_color', String), ('route_text_color', String)])
Schema([('route_id', String), ('service_id', String), ('trip_id', String), ('trip_headsign', String), ('direction_id', Int64), ('block_id', Int64), ('shape_id', String)])
Schema([('trip_id', String), ('arrival_time', String), ('departure_time', String), ('stop_id', Int64), ('stop_sequence', Int64), ('pickup_type', Int64), ('drop_off_type', Int64), ('timepoint', Int64)])
Schema([('stop_id', Int64), ('stop_name', String), ('stop_desc', String), ('stop_lat', Float64), ('stop_lon', Float64), ('zone_id', String), ('stop_url', String), ('location_type', Int64), ('parent_station', String)])
Schema([('shape_id', String), ('shape_pt_lat', Float64), ('shape_pt_lon', Float64), ('shape_pt_sequence', Int64)])


In [52]:
OUTPUT = Path("raw/processed_gtfs")
OUTPUT.mkdir(exist_ok=True)

routes.write_parquet(OUTPUT / "routes.parquet")
trips.write_parquet(OUTPUT / "trips.parquet")
stops.write_parquet(OUTPUT / "stops.parquet")
stop_times.write_parquet(OUTPUT / "stop_times.parquet")
shapes.write_parquet(OUTPUT / "shapes.parquet")
calendar.write_parquet(OUTPUT / "calendar.parquet")
calendar_dates.write_parquet(OUTPUT/"calendar_dates.parquet")

In [53]:
assert trips["trip_id"].is_unique().all()

In [54]:
assert routes["route_id"].is_unique().all()

In [55]:
assert stops["stop_id"].is_unique().all()

In [56]:
duplicates = (
    stop_times
    .group_by(["trip_id", "stop_sequence"])
    .len()
    .filter(pl.col("len") > 1)
)

print(duplicates.height)

0


In [57]:
duplicates = (
    shapes
    .group_by(["shape_id", "shape_pt_sequence"])
    .len()
    .filter(pl.col("len") > 1)
)

print(duplicates.height)

0
